# 03 - Demo Progress CabAI

**CabAI** adalah prototype klasifikasi awal penyakit daun cabai berbasis computer vision. Notebook ini disusun sebagai alur demo Jumat: masalah bisnis, data, preprocessing, model, evaluasi awal, simulasi inference, rekomendasi, kendala, dan rencana final.

## 1. Problem Statement

Petani atau pengguna non-ahli sering kesulitan mengenali gejala awal penyakit cabai dari daun. CabAI dirancang sebagai **alat bantu identifikasi awal** yang menerima foto daun, memprediksi kelas penyakit, lalu memberikan rekomendasi tindakan awal. Prototype ini tidak menggantikan ahli pertanian.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cabai.config import CLASS_NAMES, DISPLAY_NAMES, RAW_DATA_DIR, INTERIM_DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR
from cabai.data import build_manifest, assign_missing_splits, get_transforms
from cabai.model import create_model, get_device
from cabai.recommend import format_recommendation_markdown

print('Project root:', PROJECT_ROOT)
print('Classes:', CLASS_NAMES)
print('Display names:', DISPLAY_NAMES)

## 2. Dataset dan Scope

- Dataset utama final: Roboflow `Chili leaves disease classification`.
- External benchmark final: Kaggle `penyakit-cabai`.
- Scope demo/final: 5 kelas bersama.
- `powdery mildew` dikeluarkan dari eksperimen utama karena tidak tersedia di benchmark Kaggle.

In [ ]:
import pandas as pd

roboflow_rows = build_manifest(RAW_DATA_DIR / 'roboflow_chili', dataset_name='roboflow_chili')
kaggle_rows = build_manifest(RAW_DATA_DIR / 'kaggle_penyakit_cabai', dataset_name='kaggle_penyakit_cabai')
rows = roboflow_rows or kaggle_rows
dataset_name = 'roboflow_chili' if roboflow_rows else ('kaggle_penyakit_cabai fallback' if kaggle_rows else 'belum tersedia lokal')

if rows:
    rows = assign_missing_splits(rows)
    df = pd.DataFrame(rows)
    print('Dataset used in this demo:', dataset_name)
    display(pd.crosstab(df['split'], df['label']))
else:
    df = pd.DataFrame(columns=['dataset', 'split', 'label', 'path'])
    print('Dataset lokal belum tersedia. Bagian inference akan memakai simulasi rekomendasi agar alur demo tetap bisa ditunjukkan.')

## 3. Preprocessing

Preprocessing awal mengikuti praktik transfer learning ImageNet: resize ke `224x224`, konversi RGB, normalisasi ImageNet, dan augmentasi ringan pada training seperti flip, rotasi, dan color jitter moderat.

## 4. Model

Model utama sprint adalah **EfficientNet-B0**. Backbone awal menggunakan pretrained ImageNet, lalu classifier head disesuaikan untuk 5 kelas CabAI. Ini sesuai dengan kondisi dataset publik yang ukurannya moderat dan compute gratis yang terbatas.

## 5. Hasil Training Awal

Jika `02_train_baseline.ipynb` sudah dijalankan, gambar berikut akan tersedia sebagai bukti progress training dan evaluasi awal.

In [ ]:
from IPython.display import Image, display

for figure_name in ['training_curves.png', 'confusion_matrix_internal.png', 'sample_grid.png']:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        print(figure_name)
        display(Image(filename=str(figure_path)))
    else:
        print(f'{figure_name} belum ada. Jalankan notebook 01/02 terlebih dahulu atau gunakan sebagai catatan rencana demo.')

## 6. Simulasi Prototype Inference

Bagian ini memilih satu gambar contoh dari dataset jika tersedia. Jika checkpoint model belum ada, notebook tetap menampilkan simulasi rekomendasi agar alur prototype dapat dijelaskan saat progress demo.

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import torch

sample_path = Path(df.iloc[0]['path']) if not df.empty else None
if sample_path and sample_path.exists():
    image = PILImage.open(sample_path).convert('RGB')
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title('Contoh input')
else:
    image = None
    print('Tidak ada gambar lokal. Gunakan cell rekomendasi simulasi di bawah.')

In [ ]:
from IPython.display import Markdown

checkpoint_path = CHECKPOINTS_DIR / 'efficientnet_b0_demo.pt'
if image is not None and checkpoint_path.exists():
    device = get_device()
    model = create_model('efficientnet_b0', num_classes=len(CLASS_NAMES), pretrained=False)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    transform = get_transforms(train=False)
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred_idx = int(probabilities.argmax())
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probabilities[pred_idx])
    print('Prediksi:', pred_label)
    print('Confidence:', confidence)
else:
    pred_label = 'leaf spot'
    confidence = 0.83
    print('Checkpoint/gambar belum tersedia. Menampilkan simulasi prototype:', pred_label, confidence)

display(Markdown(format_recommendation_markdown(pred_label, confidence)))

## 7. Kendala Saat Ini

- Progress dimulai dari nol sehingga sprint Jumat difokuskan pada vertical slice, bukan produk final.
- Dataset publik heterogen dan dapat memiliki label noise/domain shift.
- Compute gratis Colab/Kaggle punya batas runtime dan ketersediaan GPU.
- Evaluasi eksternal Kaggle menjadi prioritas setelah training utama Roboflow stabil.

## 8. Rencana Menuju Final

- Menyelesaikan training EfficientNet-B0 dengan tuning lebih rapi.
- Membandingkan minimal satu baseline seperti ResNet50 atau MobileNetV3.
- Menjalankan external benchmark pada Kaggle yang tidak ikut training.
- Menambahkan Grad-CAM untuk explainability visual.
- Mengemas prototype menjadi Streamlit/Hugging Face Spaces sebagai AIaaS demo.
- Menyusun laporan akhir mengikuti CRISP-DM.